In [1]:
# IMPORT LIBRARIES

In [2]:
import pandas as pd
import sqlite3

In [3]:
# CONNECT DATABASE

In [4]:
conn = sqlite3.connect('ipl_data.db')
print("Database connected successfully")

Database connected successfully


In [5]:
# CREATE VIEW: RCB MATCHES ONLY

In [6]:
conn = sqlite3.connect('ipl_data.db')

conn.execute("""
CREATE VIEW IF NOT EXISTS rcb_matches_view AS 
SELECT *
FROM matches
WHERE team1 LIKE 'Royal Challengers%'
   OR team2 LIKE 'Royal Challengers%'
""")
result = pd.read_sql_query("SELECT COUNT(*) AS total FROM rcb_matches_view", conn)
print(f"rcb_matches_view created — Total matches: {result['total'][0]}")

rcb_matches_view created — Total matches: 255


In [7]:
# CREATE VIEW: RCB BATTING DELIVERIES

In [8]:
conn.execute("DROP VIEW IF EXISTS rcb_batting_view")

conn.execute("""
CREATE VIEW rcb_batting_view AS
SELECT 
    m.season,
    m.date,
    m.winner,
    d.*
FROM deliveries d
JOIN matches m ON d.match_id = m.id
WHERE (m.team1 LIKE 'Royal Challengers%'
    OR m.team2 LIKE 'Royal Challengers%')
  AND d.batting_team LIKE 'Royal Challengers%'
""")
result = pd.read_sql_query("SELECT COUNT(*) AS total FROM rcb_batting_view", conn)
print(f"rcb_batting_view created — Total deliveries: {result['total'][0]}")
print("\nSample rows:")
sample = pd.read_sql_query("SELECT season, batter, batsman_runs FROM rcb_batting_view LIMIT 5", conn)
print(sample)

rcb_batting_view created — Total deliveries: 30023

Sample rows:
    season    batter  batsman_runs
0  2007/08  R Dravid             1
1  2007/08  W Jaffer             0
2  2007/08  W Jaffer             0
3  2007/08  W Jaffer             1
4  2007/08  R Dravid             1


In [9]:
# CREATE VIEW: RCB BOWLING DELIVERIES

In [10]:
conn.execute("DROP VIEW IF EXISTS rcb_bowling_view")

conn.execute("""
CREATE VIEW rcb_bowling_view AS
SELECT 
    m.season,
    m.date,
    m.winner,
    d.*
FROM deliveries d
JOIN matches m ON d.match_id = m.id
WHERE (m.team1 LIKE 'Royal Challengers%'
    OR m.team2 LIKE 'Royal Challengers%')
  AND d.batting_team LIKE 'Royal Challengers%'
""")

result = pd.read_sql_query("SELECT COUNT(*) AS total FROM rcb_bowling_view", conn)
print(f"rcb_bowling_view created — Total deliveries: {result['total'][0]}")
print("\nSample rows:")
sample = pd.read_sql_query("SELECT season, bowler, is_wicket FROM rcb_bowling_view LIMIT 5", conn)
print(sample)

rcb_bowling_view created — Total deliveries: 30023

Sample rows:
    season    bowler  is_wicket
0  2007/08  AB Dinda          0
1  2007/08  AB Dinda          0
2  2007/08  AB Dinda          0
3  2007/08  AB Dinda          0
4  2007/08  AB Dinda          0


In [11]:
# TOP 5 BATSMEN USING rcb_batting_view

In [12]:
query = """
SELECT 
    batter,
    SUM(batsman_runs) AS total_runs
FROM rcb_batting_view
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 5
"""

top_batsmen = pd.read_sql_query(query, conn)
print("Top 5 RCB batsmen (using view):")
print(top_batsmen)

Top 5 RCB batsmen (using view):
           batter  total_runs
0         V Kohli        8014
1  AB de Villiers        4510
2        CH Gayle        3175
3    F du Plessis        1636
4      GJ Maxwell        1266


In [13]:
# TOSS ANALYSIS USING rcb_matches_view

In [14]:
query = """
SELECT 
    CASE 
        WHEN toss_winner LIKE 'Royal Challengers%' THEN 'Won Toss'
        ELSE 'Lost Toss'
    END AS toss_result,
    COUNT(*) AS total_matches,
    COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) AS wins,
    ROUND(COUNT(CASE WHEN winner LIKE 'Royal Challengers%' THEN 1 END) * 100.0 / COUNT(*), 1) AS win_pct
FROM rcb_matches_view
GROUP BY toss_result
"""

toss_analysis = pd.read_sql_query(query, conn)
print("Toss analysis (using view — no WHERE clause needed!):")
print(toss_analysis)

Toss analysis (using view — no WHERE clause needed!):
  toss_result  total_matches  wins  win_pct
0   Lost Toss            134    62     46.3
1    Won Toss            121    61     50.4


In [15]:
# LIST ALL VIEWS IN DATABASE

In [16]:
query = """
SELECT name AS view_name
FROM sqlite_master
WHERE type = 'view'
ORDER BY name
"""

views = pd.read_sql_query(query, conn)
print("Views in database:")
print(views)

Views in database:
          view_name
0  rcb_batting_view
1  rcb_bowling_view
2  rcb_matches_view


In [17]:
# DROP AND RECREATE rcb_bowling_view

In [18]:
# Step 1: Drop it
conn.execute("DROP VIEW IF EXISTS rcb_bowling_view")
print("Dropped rcb_bowling_view")

# Step 2: Verify it's gone
views_after_drop = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name", conn
)
print(f"\nViews remaining: {views_after_drop['name'].tolist()}")

# Step 3: Recreate it
conn.execute("""
CREATE VIEW rcb_bowling_view AS
SELECT 
    m.season,
    m.date,
    m.winner,
    d.*
FROM deliveries d
JOIN matches m ON d.match_id = m.id
WHERE (m.team1 LIKE 'Royal Challengers%'
    OR m.team2 LIKE 'Royal Challengers%')
  AND d.bowling_team LIKE 'Royal Challengers%'
""")
print("\nRecreated rcb_bowling_view")

# Step 4: Verify it's back
views_final = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name", conn
)
print(f"Final views: {views_final['name'].tolist()}")

Dropped rcb_bowling_view

Views remaining: ['rcb_batting_view', 'rcb_matches_view']

Recreated rcb_bowling_view
Final views: ['rcb_batting_view', 'rcb_bowling_view', 'rcb_matches_view']


In [19]:
# CLOSE CONNECTION

In [20]:
conn.close()
print("Connection closed.")

Connection closed.


## Day 12 Summary: Views

 Completed
- Created rcb_matches_view — reusable filtered match data
- Created rcb_batting_view — RCB batting with match context
- Created rcb_bowling_view — RCB bowling with match context
- Rewrote top batsmen query using view (no JOIN needed!)
- Rewrote toss analysis using view (no WHERE needed!)
- Listed all views from sqlite_master
- Practiced DROP VIEW and recreation

 Key Concepts Learned
- CREATE VIEW view_name AS SELECT ...
- DROP VIEW IF EXISTS view_name
- Views store queries, not data
- Query views exactly like tables
- sqlite_master for listing views
- Views persist across sessions

 Why Views Matter
- Write complex logic once, reuse everywhere
- Downstream queries become dramatically simpler
- Standard practice in production databases
- Great for creating "clean" datasets for reporting

 Next
Day 13 — Mini Project: Full RCB analysis using only SQL (all concepts combined)